# Autoregressive stages: Thinker and Talker

## Learning goals
- Trace one request through Thinker -> Talker -> Code2Wav
- See hidden states become an inter-stage payload
- Understand why the runner re-applies a preprocess every decode step

> **Mac track.** Every executed cell below is a pure-Python simulation that runs on any Mac with no GPU. Cells labelled **Linux GPU lab** show the *real* vLLM-Omni commands — they are shown as text and are **not** run here, because the official `vllm serve --omni` runtime is CUDA-only.

## The Qwen-Omni speech path

```text
multimodal input -> Thinker (AR text) -> text + hidden states
                 -> Talker  (AR audio codes) -> Code2Wav -> waveform
```

Thinker and Talker are *both* autoregressive, but their vocabularies differ: Thinker emits text tokens, Talker emits audio-codec tokens conditioned on Thinker's hidden states.

In [ ]:
from omni_tutorial import build_voice_pipeline
graph = build_voice_pipeline()
result, trace = graph.run('What is in this scene?')
for e in trace:
    print(f"{e['stage']:8} ({e['kind']:5}) -> {e['output']}")
print('connector transfers:', graph.connector.transfers)

## Hidden states are an internal dependency

Thinker's hidden representation is not a client response — it is conditioning for the Talker. A model-specific **preprocess** function selects, projects, and reshapes it. Crucially, the real vLLM-Omni model runner re-applies this preprocess *every decode iteration*, because the Talker fuses fresh Thinker output with its own running state at each step.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
def talker_preprocess(thinker_hidden, step):
    # project hidden state and add a per-step positional nudge (toy stand-in)
    proj = thinker_hidden @ rng.standard_normal((thinker_hidden.shape[-1], 4))
    return proj + step * 0.01
hidden = rng.standard_normal((1, 8))
for step in range(3):
    cond = talker_preprocess(hidden, step)
    print(f'decode step {step}: conditioning shape {cond.shape}, mean {cond.mean():+.3f}')

## Text tokens vs audio-codec tokens

The two AR stages produce sequences of different lengths and meanings. On video-input tasks the paper measures ~150 text tokens but ~545 audio tokens per request — the Talker simply runs far more decode steps. Remember that number; it is why the Talker is the bottleneck you scale in notebook 08.

In [ ]:
text_tokens, audio_tokens = 150, 545
print(f'Talker runs {audio_tokens/text_tokens:.1f}x more decode steps than the Thinker text head')

## Exercise

Using the trace above, which stage's output is the *client-facing* one, and which is purely inter-stage? Answer, then run the solution.

In [ ]:
# Solution
print('Inter-stage: thinker hidden states, talker audio codes (internal payloads).')
print('Client-facing: the vocoder waveform (and optionally the thinker text transcript).')

## Source lab

Trace `vllm_omni/model_executor/models/qwen3_omni/` and its deploy config `deploy/qwen3_omni_moe.yaml`. Find where the Talker's per-step preprocess consumes Thinker output.